# Monte Carlo reinforcement learning

_Artificial Intelligence — more_

**No model? Just play out whole episodes and average the rewards you actually got.**

Q-learning bootstraps off its own estimates. Monte Carlo RL (Reinforcement Learning) does something even simpler: it just averages real outcomes.

     Play a full episode to the end. Add up the discounted rewards you actually received: that is the return $u$.

---

This notebook is a practice scaffold for the **Monte Carlo reinforcement learning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

gamma = 0.9
def sample_return():
    # reach goal in 2..4 random steps: -1 per step, +10 at the goal
    steps = rng.integers(2, 5)
    u, disc = 0.0, 1.0
    for _ in range(steps):
        u += disc * (-1); disc *= gamma
    u += disc * 10
    return u

returns = [sample_return() for _ in range(2000)]
running = np.cumsum(returns) / np.arange(1, len(returns) + 1)
print("first 3 returns:", [round(r, 2) for r in returns[:3]])
print("Q-hat after   10 episodes:", round(running[9], 3))
print("Q-hat after  100 episodes:", round(running[99], 3))
print("Q-hat after 2000 episodes:", round(running[-1], 3))

## Visualize the data & results

_Blackjack: by averaging real hand outcomes, what is the value of standing on 20 against a dealer showing 6?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)

# Blackjack: estimate value of STANDING on player sum 20 vs dealer showing 6.
def draw():
    c = rng.integers(1, 14)
    return min(c, 10) if c != 1 else 1

def dealer_play(showing):
    total, ace = showing, (showing == 1)
    if ace: total += 10
    while total in range(17):                # hit on totals 0..16, stand on 17+
        c = draw()
        if c == 1 and total + 11 in range(22):
            total, ace = total + 11, True
        else:
            total += c
        while total > 21 and ace:
            total, ace = total - 10, False
    return total

def episode():
    dealer = dealer_play(6)                 # dealer reveals a 6, then hits to 17+
    if dealer > 21 or dealer in range(20): return 1.0   # bust or below 20 -> player wins
    if dealer > 20: return -1.0
    return 0.0                              # push at 20

N = 5000
rets = np.array([episode() for _ in range(N)])
running = np.cumsum(rets) / np.arange(1, N + 1)
xs = [1, 5, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000]
ys = [running[x - 1] for x in xs]

fig, ax = plt.subplots()
ax.plot(xs, ys, "-o", color="#4ea1ff", label="running estimate")
ax.axhline(running[-1], color="#7ee787", label="converged value ~ 0.70")
ax.set_xscale("log")
ax.set_xlabel("blackjack hands simulated"); ax.set_ylabel("average return (+1 win, -1 loss)")
ax.set_title("Monte Carlo value of 'stand on 20 vs dealer 6' as hands accumulate")
ax.legend()
plt.show()